In [18]:
import pandas as pd
import numpy as np

In [24]:
ws = '/data/geraugi/plural/dataset_files'
main_data_file = f'{ws}/500_xplainer_mimic_dataset.csv'
ref_data_file = f'{ws}/500_ref_xplainer_mimic_dataset.csv'
output_dir = '/data/geraugi/chexfound/dataset/training/'
train_set_file = f'{output_dir}/train.csv'
validation_set_file = f'{output_dir}/validation.csv'
testset_file = f'{output_dir}/test.csv'
main_dataset = pd.read_csv(main_data_file)
ref_data = pd.read_csv(ref_data_file)
aux_df = pd.read_csv("/data/geraugi/plural/pre_processed_data/compact_context_w_ref_id.csv")
dicom_split_df = aux_df.loc[:, ['dicom_id', 'split']].drop_duplicates(subset=['dicom_id'])

In [25]:
# stack ref_data and main_dataset
complete_dataset = pd.concat([main_dataset, ref_data], ignore_index=True)

In [ ]:
# merge complete_dataset with dicom_split_df
complete_dataset = complete_dataset.merge(dicom_split_df, left_on='dicom_id', right_on='dicom_id', how='left')
complete_dataset.head()

In [27]:
import ast
complete_dataset['disease_vector'] = complete_dataset['disease_vector'].apply(ast.literal_eval)
        
# Define the final keys (order matters). These are the 13 diseases of interest plus "No Findings".
disease_list = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Emphysema",
    "Fracture", "Hernia", "Lung Opacity", "Pleural Effusion", "Pleural Thickening",
    "Pneumonia", "Pneumothorax", "No Findings"
]
final_keys = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Lung Opacity", "Pleural Effusion",
    "Pneumonia", "Pneumothorax",
]
# get the index of the final keys in the disease_list
final_keys_indices = [disease_list.index(key) for key in final_keys]
# modify disease vector to consider only the final keys
complete_dataset['disease_vector'] = complete_dataset['disease_vector'].apply(
    lambda x: [x[i] for i in final_keys_indices]
)

In [ ]:
# create a new column per disease using the final keys
for key in final_keys:
    complete_dataset[key] = complete_dataset['disease_vector'].apply(lambda x: x[final_keys.index(key)])
complete_dataset.head()

In [30]:
train_set = complete_dataset[complete_dataset['split'] == 'train']
validation_set = complete_dataset[complete_dataset['split'] == 'validation']
test_set = complete_dataset[complete_dataset['split'] == 'test']
train_set.to_csv(train_set_file, index=False)
validation_set.to_csv(validation_set_file, index=False)
test_set.to_csv(testset_file, index=False)